# Kathakaar · Image → Video Pipeline

**What this does:**  
1. Loads your trained Kathakaar SDXL LoRA (`pytorch_lora_weights.safetensors`)  
2. Generates one cinematic image per story scene using the LoRA style  
3. Animates each image into a short video clip (Stable Video Diffusion img2vid)  
4. Adds spoken narration using gTTS (free, no key)  
5. Stitches scenes into a single MP4 — ready to plug into `/api/render`  

**Hardware:** Free Kaggle T4 (16 GB). Each img2vid clip ≈ 50 s on T4.  
**Runtime:** ~10–15 min for a 4-scene story.  

---
**Before you start:**  
- Upload your `pytorch_lora_weights.safetensors` to `/kaggle/input/kathakaar-lora/`  
  (Kaggle → your dataset → Add file)  
- Or paste the path to wherever you saved it.

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'Not found — check Runtime → Change runtime type → T4 GPU')
import os
print('Python:', sys.version.split()[0])

In [ ]:
# Install in one shot — pipe output to a log so the cell stays readable
!pip install -q \
    diffusers==0.30.3 \
    transformers==4.44.2 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    safetensors==0.4.5 \
    gTTS==2.5.1 \
    moviepy==1.0.3 \
    imageio==2.34.2 \
    imageio-ffmpeg==0.5.1 \
    Pillow==10.4.0 \
    > /tmp/install.log 2>&1 && echo 'Install OK' || (cat /tmp/install.log && echo 'Install FAILED')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURE YOUR STORY HERE
# ─────────────────────────────────────────────────────────────────────────────

LORA_PATH = '/kaggle/input/kathakaar-lora/pytorch_lora_weights.safetensors'
# or: LORA_PATH = '/kaggle/working/output/pytorch_lora_weights.safetensors'

TRIGGER_WORD = 'khvheritage'   # the trigger you trained with

# Story scenes — paste the 'narration' text from the Kathakaar /api/cinematic response
# (or write your own; keep each scene 1–2 sentences)
SCENES = [
    {
        'narration': 'Gather close, and listen well. In medieval Konark, the Sun Temple rose '
                     'as a colossal chariot of the sun god Surya, its twelve pairs of wheels '
                     'carved with extraordinary precision.',
        'caption':   'The Sun Temple — a colossal chariot of Surya',
        'image_prompt': f'{TRIGGER_WORD} style, Konark Sun Temple carved stone chariot wheel, '
                        'golden hour light, cinematic, wide shot, 8k detail',
    },
    {
        'narration': 'And so it came to pass that the entire structure was aligned to the cardinal '
                     'directions, and the first rays of sunrise would illuminate the principal deity.',
        'caption':   'Aligned to sunrise — stone calendar of the sun',
        'image_prompt': f'{TRIGGER_WORD} style, ancient Indian temple interior, rays of golden sunrise '
                        'streaming through stone doorway, cinematic lighting, dramatic',
    },
    {
        'narration': 'Now sit with this: the sculptors encoded astronomical knowledge into every stone, '
                     'each wheel a sundial, each spoke a time of day.',
        'caption':   'Each wheel a sundial — knowledge carved in stone',
        'image_prompt': f'{TRIGGER_WORD} style, intricate carved stone wheel close-up, fine sculpture, '
                        'ancient, warm amber light, macro detail',
    },
    {
        'narration': 'The question remains, and that is the answer. How did they know? '
                     'The cycle turns, as all cycles must.',
        'caption':   'The cycle turns — the story remembered',
        'image_prompt': f'{TRIGGER_WORD} style, Konark Sun Temple exterior at dusk, silhouette, '
                        'dramatic sky, cinematic wide angle',
    },
]

OUTPUT_DIR = '/kaggle/working/kathakaar_video'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Scenes configured: {len(SCENES)}')
print(f'LoRA path: {LORA_PATH}')
print(f'Output: {OUTPUT_DIR}')

## Step 1 — Generate scene images with your SDXL LoRA

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image

NEGATIVE = (
    'blurry, low quality, bad anatomy, watermark, text, signature, '
    'painting, stained glass, european art, illustration, cartoon, '
    'modern, ugly, deformed, noisy'
)

print('Loading SDXL base…')
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
).to('cuda')

print('Loading Kathakaar LoRA…')
pipe.load_lora_weights(LORA_PATH)
pipe.fuse_lora(lora_scale=0.85)   # 0.8–1.0; lower = subtler style

pipe.enable_xformers_memory_efficient_attention()

scene_images: list[Image.Image] = []
for i, scene in enumerate(SCENES):
    print(f'\nGenerating scene {i+1}/{len(SCENES)}: {scene["caption"][:50]}…')
    with torch.inference_mode():
        result = pipe(
            prompt=scene['image_prompt'],
            negative_prompt=NEGATIVE,
            num_inference_steps=30,
            guidance_scale=7.5,
            width=1024, height=576,   # 16:9 cinematic
            num_images_per_prompt=1,
        )
    img = result.images[0]
    img_path = os.path.join(OUTPUT_DIR, f'scene_{i:02d}.png')
    img.save(img_path)
    scene_images.append(img)
    print(f'  Saved → {img_path}')

# free SDXL VRAM before SVD
del pipe
torch.cuda.empty_cache()
print('\nAll scene images generated. VRAM freed for animation step.')

## Step 2 — Animate each image into a short video clip (SVD-XT)

In [ ]:
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
import numpy as np

print('Loading Stable Video Diffusion (img2vid-xt-1-1)…')
# SVD-XT runs in fp16 and uses ~12 GB VRAM on a T4 with the settings below
svd = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt-1-1',
    torch_dtype=torch.float16,
    variant='fp16',
).to('cuda')
svd.enable_model_cpu_offload()   # offload layers to CPU when not in use — saves ~2 GB

clip_paths: list[str] = []
FRAMES = 25          # 25 frames @ 6 fps = ~4 s per scene
FPS_OUT = 6

for i, img in enumerate(scene_images):
    print(f'\nAnimating scene {i+1}/{len(scene_images)}…')
    # SVD expects 1024×576 input
    img_resized = img.resize((1024, 576), Image.LANCZOS)
    with torch.inference_mode():
        frames = svd(
            image=img_resized,
            num_frames=FRAMES,
            num_inference_steps=20,   # 20 is good quality/speed balance on T4
            decode_chunk_size=4,      # keeps peak VRAM under 15 GB
            motion_bucket_id=90,      # 1–255: higher = more motion
            noise_aug_strength=0.02,
        ).frames[0]

    clip_path = os.path.join(OUTPUT_DIR, f'clip_{i:02d}.mp4')
    export_to_video(frames, clip_path, fps=FPS_OUT)
    clip_paths.append(clip_path)
    print(f'  Clip saved → {clip_path}')

del svd
torch.cuda.empty_cache()
print('\nAll clips animated.')

## Step 3 — Generate spoken narration (gTTS, free, no key)

In [ ]:
from gtts import gTTS

audio_paths: list[str] = []
for i, scene in enumerate(SCENES):
    print(f'Generating narration for scene {i+1}…')
    tts = gTTS(text=scene['narration'], lang='en', slow=False)
    audio_path = os.path.join(OUTPUT_DIR, f'narration_{i:02d}.mp3')
    tts.save(audio_path)
    audio_paths.append(audio_path)
    print(f'  Audio → {audio_path}')

print('All narrations generated.')

## Step 4 — Stitch clips + narration into one MP4

In [ ]:
from moviepy.editor import VideoFileClip, AudioFileClip, TextClip, CompositeVideoClip, concatenate_videoclips
import warnings
warnings.filterwarnings('ignore')

CAPTION_FONT_SIZE = 24
CAPTION_COLOR     = 'white'

composited: list = []
for i, (clip_path, audio_path, scene) in enumerate(zip(clip_paths, audio_paths, SCENES)):
    print(f'Compositing scene {i+1}…')
    video = VideoFileClip(clip_path)
    audio = AudioFileClip(audio_path)

    # Extend video to match audio length (loop last frame) or trim audio to clip
    duration = max(video.duration, audio.duration)
    video_fit = video.loop(duration=duration) if audio.duration > video.duration else video.subclip(0, duration)
    audio_fit = audio.subclip(0, duration)

    # Caption overlay — bottom centre, semi-transparent background
    caption = (TextClip(
                    scene['caption'],
                    fontsize=CAPTION_FONT_SIZE,
                    color=CAPTION_COLOR,
                    font='DejaVu-Sans',
                    bg_color='rgba(0,0,0,0.5)',
                    size=(video_fit.w - 40, None),
                    method='caption',
                    align='center',
               )
               .set_duration(duration)
               .set_position(('center', video_fit.h - CAPTION_FONT_SIZE * 3)))

    composed = CompositeVideoClip([video_fit, caption])
    composed = composed.set_audio(audio_fit)
    composited.append(composed)
    print(f'  Scene {i+1}: {duration:.1f} s')

print('\nConcatenating all scenes…')
final = concatenate_videoclips(composited, method='compose')

output_path = os.path.join(OUTPUT_DIR, 'kathakaar_cinematic.mp4')
final.write_videofile(
    output_path,
    fps=6,
    codec='libx264',
    audio_codec='aac',
    threads=4,
    logger='bar',
)
print(f'\n✓ Final video: {output_path}')
print(f'  Duration: {final.duration:.1f} s')
print(f'  Size: {os.path.getsize(output_path) / 1024**2:.1f} MB')

## Step 5 — Preview and download

In [ ]:
from IPython.display import Video, display
display(Video(output_path, embed=True, width=800))

In [ ]:
# Kaggle: the file appears in Output → /kaggle/working/ automatically.
# Colab: uncomment the line below to trigger a download prompt.
# from google.colab import files; files.download(output_path)
print('Download the video from the Output panel on the right →')
print(f'File: {output_path}')

## How to plug this into Kathakaar's `/api/render`

1. Download `kathakaar_cinematic.mp4` from Kaggle Output.
2. Upload it to a free hosting service:
   - **Cloudflare R2** (free 10 GB/month storage, free egress) — best option
   - **Backblaze B2** (free 10 GB storage)
   - **Google Drive** (public sharing link → direct-link proxy via `drive.google.com/uc?export=download&id=FILE_ID`)
3. The public URL is what `/api/render` returns as `video_url` in its response.
4. In `Kathakaar/studio/app/render.py`, replace the `TODO` stub with:
   ```python
   return {'status': 'done', 'video_url': 'https://your-cdn/kathakaar_cinematic.mp4'}
   ```
5. When a user clicks 'Render real video', the player will load and play your AI-generated MP4.

---

## Tips for better results

| Problem | Fix |
|---|---|
| Washed-out colours | Increase `lora_scale` from 0.85 → 1.0 |
| Too much camera shake | Lower `motion_bucket_id` from 90 → 60 |
| Generic-looking images | Improve the `image_prompt` — be specific about light, materials, angle |
| Audio out of sync | Use a constant `FPS_OUT=6` and `duration = audio.duration` strictly |
| T4 OOM on SVD | Reduce `num_frames` from 25 → 14 (= ~2.3 s clips) |

## Upgrading to ElevenLabs voice (optional)

Replace the gTTS cell with:
```python
import requests
XI_KEY = 'your_elevenlabs_key'
VOICE_ID = 'ErXwobaYiN019PkySvjV'  # 'Antoni' — calm, documentary tone
for i, scene in enumerate(SCENES):
    r = requests.post(
        f'https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}',
        headers={'xi-api-key': XI_KEY, 'Content-Type': 'application/json'},
        json={'text': scene['narration'], 'model_id': 'eleven_multilingual_v2',
              'voice_settings': {'stability': 0.5, 'similarity_boost': 0.75}},
    )
    with open(audio_paths[i], 'wb') as f:
        f.write(r.content)
```